# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Concentration Inequalities

This notebook accompanies Chapter 3, Section 3.1. We keep the hypotheses of
Markov, Chebyshev, and Hoeffding visible, distinguish one-sided and two-sided
bounds, and construct a finite-sample confidence interval for a Bernoulli
probability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

rng = np.random.default_rng(404)


## Choosing a useful bound

- **Markov:** if $X\geq0$, $\mathbb E[X]<\infty$, and $\varepsilon>0$, then
  $\mathbb P(X\geq\varepsilon)\leq\mathbb E[X]/\varepsilon$.
- **Chebyshev:** if $\mathbb E[X^2]<\infty$ and $\varepsilon>0$, then
  $\mathbb P(|X-\mathbb E[X]|\geq\varepsilon) \leq\operatorname{Var}(X)/\varepsilon^2$.
- **Hoeffding:** if $X_1,\ldots,X_n$ are independent and each is bounded
  in an interval, then the sample mean has exponential tail bounds. If all
  $a<b$, $X_i\in[a,b]$ almost surely, and $\varepsilon>0$,

  $$
  \mathbb P(|\overline X_n-\mathbb E[\overline X_n]|\geq\varepsilon)
  \leq2\exp\left(-\frac{2n\varepsilon^2}{(b-a)^2}\right).
  $$

The factor 2 comes from combining upper and lower tails. Any numerical upper
bound may be clipped at 1.


In [ ]:
# A finite Markov/Chebyshev calculation.
x_values = np.array([0.0, 2.0, 8.0])
probabilities = np.array([1 / 2, 1 / 3, 1 / 6])
mean_x = np.sum(x_values * probabilities)
variance_x = np.sum((x_values - mean_x) ** 2 * probabilities)

exact_markov_event = probabilities[x_values >= 2].sum()
markov_bound = min(1.0, mean_x / 2)
exact_chebyshev_event = probabilities[np.abs(x_values - mean_x) >= 2].sum()
chebyshev_bound = min(1.0, variance_x / 2**2)

print(f"E[X]={mean_x:.3f}, Var(X)={variance_x:.3f}")
print(f"P(X >= 2)={exact_markov_event:.3f}, clipped Markov bound={markov_bound:.3f}")
print(f"P(|X-E[X]| >= 2)={exact_chebyshev_event:.3f}, clipped Chebyshev bound={chebyshev_bound:.3f}")


## Comparing exact probabilities with bounds

Let $X_1,\ldots,X_n$ be i.i.d. $\mathrm{Bernoulli}(p)$. Then
$\mathbb E[\overline X_n]=p$ and
$\operatorname{Var}(\overline X_n)=p(1-p)/n$. Chebyshev and Hoeffding
therefore give two-sided bounds for
$\mathbb P(|\overline X_n-p|\geq\varepsilon)$. The exact probability can
be computed from the binomial PMF because $n\overline X_n\sim \mathrm{Binomial}(n,p)$.


In [ ]:
p = 0.35
epsilon = 0.10
sample_sizes = np.array([20, 50, 100, 200, 500])

exact_probabilities = []
chebyshev_bounds = []
hoeffding_bounds = []
for n in sample_sizes:
    counts = np.arange(n + 1)
    event = np.abs(counts / n - p) >= epsilon - 1e-12
    exact_probabilities.append(binom.pmf(counts[event], n, p).sum())
    chebyshev_bounds.append(min(1.0, p * (1 - p) / (n * epsilon**2)))
    hoeffding_bounds.append(min(1.0, 2 * np.exp(-2 * n * epsilon**2)))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogy(sample_sizes, exact_probabilities, "o-", label="exact two-sided tail")
ax.semilogy(sample_sizes, chebyshev_bounds, "s--", label="Chebyshev")
ax.semilogy(sample_sizes, hoeffding_bounds, "^--", label="Hoeffding")
ax.set(xlabel="n", ylabel="probability or upper bound")
ax.legend()
plt.show()


## Building a confidence interval with Hoeffding's inequality

For i.i.d. $X_i\sim\mathrm{Bernoulli}(p)$ and $0<\alpha<1$, set

$$
\delta_{n,\alpha}=\sqrt{\frac{\log(2/\alpha)}{2n}}.
$$

Then $[\overline X_n-\delta_{n,\alpha}, \overline X_n+\delta_{n,\alpha}]$ contains the fixed parameter $p$ with
probability at least $1-\alpha$. Intersecting the interval with $[0,1]$
cannot remove $p$, so it cannot reduce coverage. Coverage is a repeated-
sampling statement about a random interval, not a posterior probability for
the fixed parameter after one interval has been observed.


In [ ]:
p = 0.30
alpha = 0.05
repetitions = 5000

for n in (25, 100, 400):
    counts = rng.binomial(n, p, size=repetitions)
    estimates = counts / n
    delta = np.sqrt(np.log(2 / alpha) / (2 * n))
    lower = np.maximum(0.0, estimates - delta)
    upper = np.minimum(1.0, estimates + delta)
    coverage = np.mean((lower <= p) & (p <= upper))
    print(f"n={n:3d}: simulated coverage={coverage:.4f}, mean width={(upper-lower).mean():.4f}")


## What changes when the sample size is random?

Suppose an i.i.d. sample is filtered by a fixed event $A$, with
$\mathbb P(A)>0$, and success within the retained group has conditional
probability $q$. The retained count $N_A$ is random. Conditional on
$N_A=k\geq1$, the retained success indicators are i.i.d.
$\mathrm{Bernoulli}(q)$, so the half-width uses $k$, not the original
sample size. When $N_A=0$, the notes define the conservative interval
$[0,1]$.


In [ ]:
original_n = 80
p_a = 0.05
q = 0.40
in_a = rng.random(original_n) < p_a
success = in_a & (rng.random(original_n) < q)
n_a = int(in_a.sum())

if n_a == 0:
    interval = (0.0, 1.0)
    estimate = None
else:
    estimate = success.sum() / n_a
    delta = np.sqrt(np.log(2 / alpha) / (2 * n_a))
    interval = (max(0.0, estimate - delta), min(1.0, estimate + delta))

print("retained N_A =", n_a)
print("conditional-probability estimate =", estimate)
print("95% Hoeffding interval =", interval)


## Try it yourself

1. For each situation, name the bound whose hypotheses are available:
   (a) one nonnegative variable with a finite mean; (b) one variable with a
   finite variance; (c) an average of independent variables in $[-2,3]$.
2. Derive the one-sided and two-sided Hoeffding bounds for variables in
   $[-2,3]$, keeping track of the interval length and the factor 2.
3. Repeat the coverage experiment with $p=0.02$. Compare unclipped and
   clipped widths, and explain why clipping does not reduce coverage.
